In [28]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV

from bs4 import BeautifulSoup
import re

In [29]:
df = pd.read_csv("./data/IMDB Dataset.csv")

In [30]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [31]:
df.sample(5)

,review,sentiment
26296,"What of Domino did I hate over everything, and...",negative
12652,Lets put it this way. I actually get this movi...,negative
15250,This is a textbook example of how Hollywood di...,negative
27643,"First off, I want to say, ""Thanks, Disney, for...",positive
7900,It's great how this movie pulls you along. I h...,positive


In [32]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg ])

In [33]:
from sklearn.model_selection import train_test_split


In [34]:
train,test = train_test_split(df_reviews,test_size =0.33,random_state=42)

In [35]:
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

In [36]:
train_y.value_counts()

sentiment
negative    3378
positive    3322
Name: count, dtype: int64

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

train_x_vector = tfidf.fit_transform(train_x)

In [39]:
train_x.shape

(6700,)

In [40]:
train_x_vector.shape

(6700, 44414)

In [41]:
type(train_x_vector)

scipy.sparse._csr.csr_matrix

In [42]:
primera_resenia = pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out()).iloc[0]

In [43]:
primera_resenia

00           NaN
000          NaN
007          NaN
00am         NaN
00s          NaN
            ... 
ísnt         NaN
île          NaN
önsjön       NaN
über         NaN
überwoman    NaN
Name: 6746, Length: 44414, dtype: Sparse[float64, nan]

In [44]:
train_x.iloc[0]

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

In [45]:
primera_resenia[primera_resenia != 0]

00           NaN
000          NaN
007          NaN
00am         NaN
00s          NaN
            ... 
ísnt         NaN
île          NaN
önsjön       NaN
über         NaN
überwoman    NaN
Name: 6746, Length: 44414, dtype: Sparse[float64, nan]

In [46]:
train_x.iloc[0]

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

In [47]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)
test_x_vector = tfidf.transform(test_x)

In [48]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))

['positive']
['positive']
['negative']


In [ ]:
print(svc.score(test_x_vector, test_y))



0.8809090909090909


NameError: name 'precision_svc' is not defined

In [50]:
from sklearn.metrics import f1_score

f1_score(test_y,svc.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

array([0.88265154, 0.87911412])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


modelos = {
    'Regresión Logística': {
            'modelo': LogisticRegression(),
            'parametros': {
                'C': [0.01, 0.1, 1, 10, 100],
                'penalty': ['l1', 'l2'],
                'solver': ['liblinear', 'saga'],
                'max_iter': [100, 500, 1000]
            }
        },

   'Clasificador de Árbol de Decisión': {
           'modelo': DecisionTreeClassifier(),
           'parametros': {
               'splitter': ['best', 'random'],
               'max_depth': [None, 1, 2, 3, 4]
           }
        },

    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros':{}
    }
}

train_x_dense = train_x_vector.toarray()
test_x_dense = test_x_vector.toarray()

# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}
   
# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    # Inicializar GridSearchCV con el modelo y los hiperparámetros
    grid_search = GridSearchCV(
        estimator=info_modelo['modelo'],
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1,
    )

    #para gaussiannb que utiliza matriz densa
    if nombre == 'GaussianNB':
        X_tr, X_te = train_x_dense, test_x_dense
    else:
        X_tr, X_te = train_x_vector, test_x_vector


    # Ajustar GridSearchCV con los datos de entrenamiento
    grid_search.fit(X_tr, train_y)
    
    # Hacer predicciones con el modelo ajustado
    y_pred = grid_search.predict(X_te)

    
    # Calcular la precisión de las predicciones
    precision = accuracy_score(test_y, y_pred)
    
    # Almacenar los resultados del modelo
    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })

    estimadores[nombre] = grid_search.best_estimator_
    
    # Actualizar el mejor modelo si la precisión actual es mayor que la mejor precisión encontrada
    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")
           

c:\Users\Carlos\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Rendimiento de los modelos de clasificación
                              Modelo  Precisión
0                Regresión Logística       0.88
1  Clasificador de Árbol de Decisión       0.71
2                         GaussianNB       0.64
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Regresión Logística
Precisión: 0.88


In [51]:
from sklearn.metrics import classification_report

print(classification_report(test_y,
                            svc.predict(test_x_vector),
                            labels = ['positive','negative']))

              precision    recall  f1-score   support

    positive       0.88      0.88      0.88      1678
    negative       0.88      0.88      0.88      1622

    accuracy                           0.88      3300
   macro avg       0.88      0.88      0.88      3300
weighted avg       0.88      0.88      0.88      3300



In [52]:
from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(test_y,
                           svc.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat

array([[1478,  200],
       [ 193, 1429]])

# ANALISIS LR, RF Y GAUSS


In [59]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


modelos = {
    'Regresión Logística': {
            'modelo': LogisticRegression(),
            'parametros': {
                'C': [0.01, 0.1, 1, 10, 100],
                'penalty': ['l1', 'l2'],
                'solver': ['liblinear', 'saga'],
                'max_iter': [100, 500, 1000]
            }
        },

   'Clasificador de Árbol de Decisión': {
           'modelo': DecisionTreeClassifier(),
           'parametros': {
               'splitter': ['best', 'random'],
               'max_depth': [None, 1, 2, 3, 4]
           }
        },

    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros':{}
    }
}

train_x_dense = train_x_vector.toarray()
test_x_dense = test_x_vector.toarray()

# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}
   
# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    # Inicializar GridSearchCV con el modelo y los hiperparámetros
    grid_search = GridSearchCV(
        estimator=info_modelo['modelo'],
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1,
    )

    #para gaussiannb que utiliza matriz densa
    if nombre == 'GaussianNB':
        X_tr, X_te = train_x_dense, test_x_dense
    else:
        X_tr, X_te = train_x_vector, test_x_vector


    # Ajustar GridSearchCV con los datos de entrenamiento
    grid_search.fit(X_tr, train_y)
    
    # Hacer predicciones con el modelo ajustado
    y_pred = grid_search.predict(X_te)

    
    # Calcular la precisión de las predicciones
    precision = accuracy_score(test_y, y_pred)
    
    # Almacenar los resultados del modelo
    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })

    estimadores[nombre] = grid_search.best_estimator_
    
    # Actualizar el mejor modelo si la precisión actual es mayor que la mejor precisión encontrada
    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")
           

c:\Users\Carlos\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Rendimiento de los modelos de clasificación
                              Modelo  Precisión
0                Regresión Logística       0.88
1  Clasificador de Árbol de Decisión       0.71
2                         GaussianNB       0.64
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Regresión Logística
Precisión: 0.88
